In [1]:
import sys
from pathlib import Path
import mlflow

sys.executable

# Repo root (notebook is in /notebooks)
PROJECT_ROOT = Path.cwd().parent

# Dataset path
csv_path = PROJECT_ROOT / "data" / "raw" / "adult-census.csv"

# Everything under repo/mlflow/
MLFLOW_DIR = PROJECT_ROOT / "mlflow"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_DB = (MLFLOW_DIR / "mlflow.db").resolve()
ARTIFACT_ROOT = (MLFLOW_DIR / "artifacts").resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")

EXPERIMENT_NAME = "adult_census_tracking"
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=ARTIFACT_ROOT.as_uri(),
    )
mlflow.set_experiment(EXPERIMENT_NAME)

# Skore outputs (local, then logged to MLflow as artifacts)
SKORE_DIR = PROJECT_ROOT / "reports" / "skore"
SKORE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT  =", PROJECT_ROOT)
print("csv_path     =", csv_path)
print("exists?      =", csv_path.exists())
print("MLFLOW_DIR    =", MLFLOW_DIR)
print("TRACKING_URI  =", mlflow.get_tracking_uri())
print("MLFLOW_DB     =", MLFLOW_DB)
print("ARTIFACT_ROOT =", ARTIFACT_ROOT)
print("SKORE_DIR     =", SKORE_DIR)

2026/01/14 13:10:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/14 13:10:30 INFO mlflow.store.db.utils: Updating database tables
2026/01/14 13:10:30 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/14 13:10:30 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/14 13:10:30 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/14 13:10:30 INFO alembic.runtime.migration: Will assume non-transactional DDL.


PROJECT_ROOT  = H:\Documents\2. Perso\github\mlflow
csv_path     = H:\Documents\2. Perso\github\mlflow\data\raw\adult-census.csv
exists?      = True
MLFLOW_DIR    = H:\Documents\2. Perso\github\mlflow\mlflow
TRACKING_URI  = sqlite:///H:/Documents/2. Perso/github/mlflow/mlflow/mlflow.db
MLFLOW_DB     = H:\Documents\2. Perso\github\mlflow\mlflow\mlflow.db
ARTIFACT_ROOT = H:\Documents\2. Perso\github\mlflow\mlflow\artifacts
SKORE_DIR     = H:\Documents\2. Perso\github\mlflow\reports\skore


In [2]:
import pandas as pd

adult_census = pd.read_csv(csv_path)
adult_census = adult_census.drop(columns="education.num")

target_name = "income"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name])

In [3]:
from sklearn.compose import make_column_selector as selector

numerical_columns_selector = selector(dtype_exclude=object)
categorical_columns_selector = selector(dtype_include=object)

numerical_columns = numerical_columns_selector(data)
categorical_columns = categorical_columns_selector(data)

In [4]:
from sklearn.model_selection import train_test_split

# 1) Encode target with the exact mapping you want
target_map = {"<=50K": 0, ">50K": 1}

# if target is a pandas Series of strings
target_enc = target.map(target_map).astype("int64")

# (optional) safety check
# assert target_enc.isna().sum() == 0, f"Unmapped labels: {target[target_enc.isna()].unique()}"

# 2) Train/test split
data_train, data_test, target_train, target_test = train_test_split(
    data, target_enc, test_size=0.2, random_state=42, stratify=target_enc
)

# Model #1 — LogisticRegression

In [5]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler()

preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    (numerical_preprocessor, numerical_columns),
)

logreg_ohe_scaler = make_pipeline(preprocessor, LogisticRegression(max_iter=500))

In [6]:
%%time
_ = logreg_ohe_scaler.fit(data_train, target_train)

logreg_ohe_scaler.score(data_test, target_test)

CPU times: total: 266 ms
Wall time: 299 ms


0.8544449562413634

In [7]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="logreg_ohe_scaler_cv"):
    cv_results = cross_validate(logreg_ohe_scaler, data, target_enc, cv=5)
    scores = cv_results["test_score"]

    mean_logreg = float(scores.mean())
    std_logreg = float(scores.std())

    mlflow.log_param("model_key", "logreg_ohe_scaler")
    mlflow.log_param("model_family", "LogisticRegression")
    mlflow.log_param("categorical_encoder", "OneHotEncoder(ignore)")
    mlflow.log_param("numerical_scaler", "StandardScaler")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("cv_accuracy_mean", mean_logreg)
    mlflow.log_metric("cv_accuracy_std", std_logreg)

    print(f"The mean cross-validation accuracy is: {mean_logreg:.3f} ± {std_logreg:.3f}")

The mean cross-validation accuracy is: 0.826 ± 0.031


# Model #2 — HistGradientBoosting

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)

preprocessor_hgb = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    remainder="passthrough",
)

hgb_ordinal = make_pipeline(preprocessor_hgb, HistGradientBoostingClassifier())

In [9]:
%%time
_ = hgb_ordinal.fit(data_train, target_train)

hgb_ordinal.score(data_test, target_test)

CPU times: total: 1.7 s
Wall time: 4.9 s


0.8681099339781975

In [10]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="hgb_ordinal_cv"):
    cv_results = cross_validate(hgb_ordinal, data, target_enc, cv=5)
    scores = cv_results["test_score"]

    mean_hgb = float(scores.mean())
    std_hgb = float(scores.std())

    mlflow.log_param("model_key", "hgb_ordinal")
    mlflow.log_param("model_family", "HistGradientBoostingClassifier")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder(unknown=-1)")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("cv_accuracy_mean", mean_hgb)
    mlflow.log_metric("cv_accuracy_std", std_hgb)

    print(f"The mean cross-validation accuracy is: {mean_hgb:.3f} ± {std_hgb:.3f}")

The mean cross-validation accuracy is: 0.811 ± 0.030


# Model #3 — RandomForest

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline

rf_ohe = make_pipeline(
    preprocessor,  # le preprocessor OHE+scaler du model 1
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
)

In [12]:
%%time
_ = rf_ohe.fit(data_train, target_train)

rf_ohe.score(data_test, target_test)

CPU times: total: 2min 3s
Wall time: 36.2 s


0.8524489482573315

In [13]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="rf_ohe_cv"):
    cv_results = cross_validate(rf_ohe, data, target_enc, cv=5)
    scores = cv_results["test_score"]

    mean_rf = float(scores.mean())
    std_rf = float(scores.std())

    mlflow.log_param("model_key", "rf_ohe")
    mlflow.log_param("model_family", "RandomForestClassifier")
    mlflow.log_param("categorical_encoder", "OneHotEncoder(ignore)")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("cv_accuracy_mean", mean_rf)
    mlflow.log_metric("cv_accuracy_std", std_rf)

    print(f"The mean cross-validation accuracy is: {mean_rf:.3f} ± {std_rf:.3f}")

The mean cross-validation accuracy is: 0.808 ± 0.021


# Model #4 — XGBoost

In [14]:
from xgboost import XGBClassifier
from sklearn.pipeline import make_pipeline

xgb_ohe = make_pipeline(
    preprocessor,  # OHE+scaler
    XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
    ),
)

In [15]:
%%time
_ = xgb_ohe.fit(data_train, target_train)

xgb_ohe.score(data_test, target_test)

CPU times: total: 4.06 s
Wall time: 1.14 s


0.8674957776754184

In [16]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="xgb_ohe_cv"):
    cv_results = cross_validate(xgb_ohe, data, target_enc, cv=5)
    scores = cv_results["test_score"]

    mean_xgb = float(scores.mean())
    std_xgb = float(scores.std())

    mlflow.log_param("model_key", "xgb_ohe")
    mlflow.log_param("model_family", "XGBClassifier")
    mlflow.log_param("categorical_encoder", "OneHotEncoder(ignore)")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("eval_metric", "logloss")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("cv_folds", 5)
    
    mlflow.log_metric("cv_accuracy_mean", mean_xgb)
    mlflow.log_metric("cv_accuracy_std", std_xgb)

    print(f"The mean cross-validation accuracy is: {mean_xgb:.3f} ± {std_xgb:.3f}")

The mean cross-validation accuracy is: 0.818 ± 0.025


# Collect CV means + select the best model key

In [17]:
cv_means = {
    "logreg_ohe_scaler": mean_logreg,
    "hgb_ordinal": mean_hgb,
    "rf_ohe": mean_rf,
    "xgb_ohe": mean_xgb,
}

best_key = max(cv_means, key=cv_means.get)
best_key

'logreg_ohe_scaler'

# Map keys to model objects + pick the champion model

In [18]:
models = {
    "logreg_ohe_scaler": logreg_ohe_scaler,
    "hgb_ordinal": hgb_ordinal,
    "rf_ohe": rf_ohe,
    "xgb_ohe": xgb_ohe,
}

best_model = models[best_key]

# Champion run (refit + test + log_model)

In [19]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name=f"{best_key}_champion"):
    # Refit on train split
    best_model.fit(data_train, target_train)

    # Final sanity check on test split
    test_acc = best_model.score(data_test, target_test)

    mlflow.log_param("selected_model", best_key)
    mlflow.log_param("selection_metric", "cv_accuracy_mean")
    mlflow.log_metric("test_accuracy", float(test_acc))

    # Store the model artifact (this is the model you will register later)
    mlflow.sklearn.log_model(best_model, name="model")

    print("Champion:", best_key)
    print("Test accuracy:", float(test_acc))

Champion: logreg_ohe_scaler
Test accuracy: 0.8544449562413634


# Skore — ComparisonReport + plots

In [23]:
from sklearn.base import clone
from skore import EstimatorReport, ComparisonReport

# Build reports (Skore refits internally; clone to avoid state issues)
ds = dict(
    X_train=data_train, y_train=target_train,
    X_test=data_test,   y_test=target_test,
)

reports = {
    name: EstimatorReport(clone(estimator), **ds)
    for name, estimator in models.items()
}

comparison = ComparisonReport(reports)

# Save under repo/reports/skore (uses SKORE_DIR from your setup cell)
out_dir = SKORE_DIR
out_dir.mkdir(parents=True, exist_ok=True)

# --- Native Skore comparison plots (single plot comparing all models) ---
roc_disp = comparison.metrics.roc(data_source="test")
roc_disp.plot()
roc_path = out_dir / "comparison_roc_test.png"
roc_disp.figure_.savefig(roc_path, dpi=200)
mlflow.log_artifact(str(roc_path))

pr_disp = comparison.metrics.precision_recall(data_source="test")
pr_disp.plot()
pr_path = out_dir / "comparison_pr_test.png"
pr_disp.figure_.savefig(pr_path, dpi=200)
mlflow.log_artifact(str(pr_path))

# --- Optional: log a metrics table as CSV (easy to diff) ---
metrics_df = comparison.metrics.summarize(data_source="test").frame()
csv_path = out_dir / "comparison_metrics_test.csv"
metrics_df.to_csv(csv_path, index=False)
mlflow.log_artifact(str(csv_path))

print("Saved to:", out_dir)
print("Logged skore artifacts:", roc_path.name, pr_path.name, csv_path.name)

Output()

Output()

Output()

Saved to: H:\Documents\2. Perso\github\mlflow\reports\skore
Logged skore artifacts: comparison_roc_test.png comparison_pr_test.png comparison_metrics_test.csv


In [24]:
import mlflow
import mlflow.sklearn

REGISTERED_MODEL_NAME = "adult_census_classifier"

with mlflow.start_run(run_name=f"{best_key}_register", nested=True):
    best_model.fit(data_train, target_train)

    mlflow.sklearn.log_model(
        best_model,
        name="model",
        registered_model_name=REGISTERED_MODEL_NAME,
    )

print("Registered:", REGISTERED_MODEL_NAME)

2026/01/14 13:23:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/14 13:23:23 INFO mlflow.store.db.utils: Updating database tables
2026/01/14 13:23:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/14 13:23:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Successfully registered model 'adult_census_classifier'.
Created version '1' of model 'adult_census_classifier'.


Registered: adult_census_classifier
